# AudDSR: Two-Stage Anomaly Detection
Discrete VQ-VAE + Anomaly Detector pipeline for cross-machine generalization

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from scipy.io import wavfile
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Configuration

In [ ]:
# Quick mode for testing
QUICK_TEST_MODE = True
STAGE1_EPOCHS = 2 if QUICK_TEST_MODE else 5
STAGE2_EPOCHS = 2 if QUICK_TEST_MODE else 5

# Quantizer configuration
QUANTIZER_EMBEDDINGS = 512  # Use 'large' capacity: 512 embeddings

# Mel-spectrogram configuration ("balanced" frontend)
MEL_CONFIG = {
    "n_fft": 1024,
    "hop_length": 256,
    "n_mels": 128,
    "norm": "per_bin"
}

# Anomaly mask configuration
CORRUPTION_CONFIG = {
    "mode": "mixed_structured",
    "ratio": 0.28,
    "block_hw": (8, 8),
    "replace_strategy": "batch_mix+random",
    "mix_alpha": 0.65,
    "mask_dilation": 2,
}

# Detector configuration
DETECTOR_WIDTH = 64  # "wide" detector

# Data configuration
DATA_ROOT = r"C:\Users\20223669\OneDrive - TU Eindhoven\Desktop\Soroma Internship\15097779"
ALL_MACHINES = ["bearing", "fan", "gearbox", "slider", "ToyCar", "ToyTrain", "valve"]

print(f"Using {STAGE1_EPOCHS} epochs for Stage 1")
print(f"Using {STAGE2_EPOCHS} epochs for Stage 2")

## Audio Processing

In [ ]:
def load_audio(file_path, sr=16000, mono=True, duration=None):
    """Load audio file with resampling."""
    waveform, orig_sr = torchaudio.load(file_path)
    
    if mono and waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    if orig_sr != sr:
        resampler = T.Resample(orig_sr, sr)
        waveform = resampler(waveform)
    
    if duration:
        samples = int(duration * sr)
        if waveform.shape[1] > samples:
            waveform = waveform[:, :samples]
        elif waveform.shape[1] < samples:
            waveform = F.pad(waveform, (0, samples - waveform.shape[1]))
    
    return waveform.squeeze(0)

def compute_mel(waveform, config):
    """Compute mel-spectrogram."""
    mel_transform = T.MelSpectrogram(
        sample_rate=16000,
        n_fft=config["n_fft"],
        hop_length=config["hop_length"],
        n_mels=config["n_mels"],
    )
    mel = mel_transform(waveform)
    mel = torch.log(mel + 1e-9)  # Log scale
    return mel

## Dataset

In [ ]:
class AnomalyDataset(Dataset):
    """Load audio files for anomaly detection training."""
    def __init__(self, file_list, mel_config, duration=10.0):
        self.file_list = file_list
        self.mel_config = mel_config
        self.duration = duration
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        waveform = load_audio(file_path, duration=self.duration)
        mel = compute_mel(waveform, self.mel_config)
        return mel

def get_file_list(machine_type, data_root, split="normal"):
    """Get list of audio files for a machine type."""
    base_path = Path(data_root) / machine_type
    if not base_path.exists():
        print(f"Warning: {base_path} not found")
        return []
    
    if split == "normal":
        pattern = "**/*normal*"
    elif split == "anomaly":
        pattern = "**/*anomaly*"
    else:  # all
        pattern = "**/*.wav"
    
    return sorted([str(f) for f in base_path.glob(pattern) if f.suffix == ".wav"])

## Stage 1: Discrete VQ-VAE

In [ ]:
class VectorQuantizer(nn.Module):
    """Vector Quantizer module."""
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        nn.init.uniform_(self.embeddings.weight, -1.0 / num_embeddings, 1.0 / num_embeddings)
    
    def forward(self, x):
        """Quantize input."""
        # x: (B, C, H, W)
        x_flat = x.reshape(-1, self.embedding_dim)
        distances = torch.cdist(x_flat, self.embeddings.weight)  # (N, K)
        indices = torch.argmin(distances, dim=1)
        quantized = self.embeddings(indices).reshape(x.shape)
        return quantized, indices

class VQVAE(nn.Module):
    """VQ-VAE for learning discrete representations."""
    def __init__(self, in_channels=1, num_embeddings=512):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),
            nn.ReLU(),
        )
        self.vq = VectorQuantizer(num_embeddings, 64)
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, in_channels, 4, stride=2, padding=1),
        )
    
    def forward(self, x):
        z = self.encoder(x)
        z_q, indices = self.vq(z)
        recon = self.decoder(z_q)
        return recon, z, z_q

def train_stage1(machine_list, data_root, mel_config, epochs=2, batch_size=16):
    """Train Stage 1: VQ-VAE on all machines."""
    model = VQVAE(in_channels=1, num_embeddings=QUANTIZER_EMBEDDINGS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    for epoch in range(epochs):
        total_loss = 0
        for machine in machine_list:
            files = get_file_list(machine, data_root, split="normal")
            if not files:
                continue
            
            dataset = AnomalyDataset(files, mel_config)
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
            
            for mel_batch in loader:
                mel_batch = mel_batch.unsqueeze(1).to(DEVICE)  # (B, 1, F, T)
                optimizer.zero_grad()
                recon, z, z_q = model(mel_batch)
                
                recon_loss = F.mse_loss(recon, mel_batch)
                commit_loss = F.mse_loss(z.detach(), z_q) * 0.25
                loss = recon_loss + commit_loss
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
        
        print(f"Stage 1 - Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")
    
    return model

## Stage 2: Anomaly Detector

In [ ]:
class AnomalyDetector(nn.Module):
    """Anomaly detector on top of VQ-VAE."""
    def __init__(self, width=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        return self.net(x)

def train_stage2(vqvae, machine_list, data_root, mel_config, epochs=2, batch_size=16):
    """Train Stage 2: Anomaly detector on all machines."""
    detector = AnomalyDetector(width=DETECTOR_WIDTH).to(DEVICE)
    optimizer = torch.optim.Adam(detector.parameters(), lr=1e-3)
    
    for epoch in range(epochs):
        total_loss = 0
        for machine in machine_list:
            normal_files = get_file_list(machine, data_root, split="normal")
            anomaly_files = get_file_list(machine, data_root, split="anomaly")
            
            if not normal_files or not anomaly_files:
                continue
            
            # Create labels: 0 for normal, 1 for anomaly
            normal_dataset = AnomalyDataset(normal_files, mel_config)
            anomaly_dataset = AnomalyDataset(anomaly_files, mel_config)
            
            normal_loader = DataLoader(normal_dataset, batch_size=batch_size, shuffle=True)
            anomaly_loader = DataLoader(anomaly_dataset, batch_size=batch_size, shuffle=True)
            
            for normal_batch, anomaly_batch in zip(normal_loader, anomaly_loader):
                normal_batch = normal_batch.unsqueeze(1).to(DEVICE)
                anomaly_batch = anomaly_batch.unsqueeze(1).to(DEVICE)
                
                with torch.no_grad():
                    _, z_normal, _ = vqvae(normal_batch)
                    _, z_anomaly, _ = vqvae(anomaly_batch)
                
                # Pool over spatial dimensions
                z_normal_pool = z_normal.mean(dim=(2, 3))  # (B, C)
                z_anomaly_pool = z_anomaly.mean(dim=(2, 3))
                
                optimizer.zero_grad()
                pred_normal = detector(z_normal_pool)
                pred_anomaly = detector(z_anomaly_pool)
                
                loss = F.binary_cross_entropy(pred_normal, torch.zeros_like(pred_normal)) + \
                       F.binary_cross_entropy(pred_anomaly, torch.ones_like(pred_anomaly))
                
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
        
        print(f"Stage 2 - Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")
    
    return detector

## Training Pipeline

In [ ]:
# Train Stage 1
print("\n=== STAGE 1: Training VQ-VAE ===")
vqvae = train_stage1(ALL_MACHINES, DATA_ROOT, MEL_CONFIG, epochs=STAGE1_EPOCHS)

# Train Stage 2
print("\n=== STAGE 2: Training Anomaly Detector ===")
detector = train_stage2(vqvae, ALL_MACHINES, DATA_ROOT, MEL_CONFIG, epochs=STAGE2_EPOCHS)

print("\n✓ Training complete!")

## Evaluation

In [ ]:
def evaluate_detector(vqvae, detector, machine_type, data_root, mel_config):
    """Evaluate detector on a machine type."""
    normal_files = get_file_list(machine_type, data_root, split="normal")
    anomaly_files = get_file_list(machine_type, data_root, split="anomaly")
    
    if not normal_files or not anomaly_files:
        print(f"Skipping {machine_type}: missing data")
        return None
    
    all_preds = []
    all_labels = []
    
    # Normal samples
    normal_dataset = AnomalyDataset(normal_files, mel_config)
    normal_loader = DataLoader(normal_dataset, batch_size=16, shuffle=False)
    
    with torch.no_grad():
        for mel_batch in normal_loader:
            mel_batch = mel_batch.unsqueeze(1).to(DEVICE)
            _, z, _ = vqvae(mel_batch)
            z_pool = z.mean(dim=(2, 3))
            scores = detector(z_pool).cpu().numpy().flatten()
            all_preds.extend(scores)
            all_labels.extend([0] * len(scores))
    
    # Anomaly samples
    anomaly_dataset = AnomalyDataset(anomaly_files, mel_config)
    anomaly_loader = DataLoader(anomaly_dataset, batch_size=16, shuffle=False)
    
    with torch.no_grad():
        for mel_batch in anomaly_loader:
            mel_batch = mel_batch.unsqueeze(1).to(DEVICE)
            _, z, _ = vqvae(mel_batch)
            z_pool = z.mean(dim=(2, 3))
            scores = detector(z_pool).cpu().numpy().flatten()
            all_preds.extend(scores)
            all_labels.extend([1] * len(scores))
    
    auc = roc_auc_score(all_labels, all_preds)
    return auc

# Evaluate on all machines
print("\n=== EVALUATION ===")
results = {}
for machine in ALL_MACHINES:
    auc = evaluate_detector(vqvae, detector, machine, DATA_ROOT, MEL_CONFIG)
    if auc is not None:
        results[machine] = auc
        print(f"{machine:12s}: AUC = {auc:.4f}")

if results:
    print(f"\nMean AUC: {np.mean(list(results.values())):.4f}")